# 阶段0只读核查

本笔记本复核冻结哈希、完整路径费用及参考比值，供技术人员重复检查。原结果不修改。完整约束与源表复算程序为同目录的独立核查.py。

## 数据与方法

采用项目专用环境。研究范围为三个单元；费用口径为增量现金流现值，固定年度分母不含规划储能动作。

In [1]:
from pathlib import Path
import json,hashlib,csv,math
audit_dir=Path.cwd()
if not (audit_dir/'证据').is_dir():
    audit_dir=Path('研究报告/00审查').resolve()
root=audit_dir.parents[1]
snapshot=json.loads((audit_dir/'证据/输入与原结果快照.json').read_text())
assert all(hashlib.sha256((root/r['path']).read_bytes()).hexdigest()==r['sha256'] for r in snapshot)
print('原输入及原结果快照核验：',len(snapshot),'文件通过')

原输入及原结果快照核验： 491 文件通过


## 结果

从原CSV重算年度比值和路径费用加总，不引入新的模型解。

In [2]:
result=root/'实验/研究/rebuild_2026/source_audit/capacity_release_simulation/reserve_policy_v4'
def read(p):
    with p.open(encoding='utf-8-sig') as f:return list(csv.DictReader(f))
summary=read(result/'summary.csv')
for r in summary:
    year=read(result/f"{r['scheme']}_{r['study_region_id']}_{r['voltage_kv']}_years.csv")
    assert math.isclose(sum(float(y['year_lifecycle_npv_10k_cny']) for y in year),float(r['objective_npv_10k_cny']),abs_tol=1e-3)
    for y in year:
        assert math.isclose(float(y['selected_capacity_mva'])/float(y['synchronous_forward_peak_mw']),float(y['actual_clr']),abs_tol=1e-7)
print('六路径费用与24行方案年度比值通过')

六路径费用与24行方案年度比值通过


In [3]:
evidence=json.loads((audit_dir/'证据/本阶段独立核查.json').read_text())
assert evidence['status']=='PASS'
print('源表、约束与现金流独立检查：',evidence['checks'],'项通过')
reproduction=json.loads((audit_dir/'证据/本阶段完整重求对账.json').read_text())
assert reproduction['status']=='PASS'
print('完整重求对账：',reproduction['checks'],'项通过')

源表、约束与现金流独立检查： 6766 项通过
完整重求对账： 1062 项通过


## 使用范围

核查通过表示当前数据与实现一致。推荐值仍受模型假设约束，不能代替完整标准承载力评估及工程实施校核。